In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

Data preparation

In [2]:
import importlib
import eval_utils

importlib.reload(eval_utils)

<module 'eval_utils' from '/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py'>

In [4]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.37it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 5932.54it/s]


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

In [4]:
import time
import pandas as pd
from langchain_community.vectorstores import FAISS

vector_storage = FAISS.from_documents(chunks, embedding)


In [5]:
sim_rt_lst = []

k_values = [4, 6, 8, 10, 12]

for k in k_values:
    retriever = vector_storage.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    sim_rt_lst.append(retriever)

In [ ]:
mmr_rt_lst = []

for k in k_values:
    retriever = vector_storage.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k,
                       "lambda_mult": 0.7}
    )
    mmr_rt_lst.append(retriever)

In [ ]:
# build chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.  
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])
# save chains with similarity retriever
sim_rag_chains = []

for retriever in sim_rt_lst:
    rag_chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )
    sim_rag_chains.append(rag_chain)

# save chains with mmr retriever
mmr_rag_chains = []

for retriever in mmr_rt_lst:
    rag_chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )
    mmr_rag_chains.append(rag_chain)

In [8]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rt_times = []
    dataset = []

    for query, reference in zip(queries, references):

        t0 = time.perf_counter()
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        t1 = time.perf_counter()

        response = rag_chain.invoke(query)
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rt_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rt_times

In [9]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

In [10]:
# select metrics
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         Faithfulness(),]

Testing

In [18]:
data_path = "evaluate_dataset/test_dataset.csv"

sim_results_lst = []
for retriever, rag_chain in zip(sim_rt_lst, sim_rag_chains):
    eval_ds, retrieval_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    sim_results_lst.append((result,retrieval_time))

Evaluating: 100%|██████████| 160/160 [17:27<00:00,  6.55s/it]


In [ ]:
for idx, (result, retrieval_time) in enumerate(sim_results_lst):
    print(f'top-{k_values[idx]}', result)
    df = result.to_pandas()
    df["retrieval_time"] = retrieval_time
    df.to_csv(f"./evaluate_results/04_retrieval_test/similarity_retrieval/top_{k_values[idx]}.csv")

top-4 {'llm_context_precision_with_reference': 0.6146, 'context_recall': 0.4679, 'nv_accuracy': 0.4000, 'faithfulness': 0.6197}
top-6 {'llm_context_precision_with_reference': 0.5943, 'context_recall': 0.5101, 'nv_accuracy': 0.4437, 'faithfulness': 0.7512}
top-8 {'llm_context_precision_with_reference': 0.6139, 'context_recall': 0.5493, 'nv_accuracy': 0.4500, 'faithfulness': 0.7991}
top-10 {'llm_context_precision_with_reference': 0.5865, 'context_recall': 0.5618, 'nv_accuracy': 0.4313, 'faithfulness': 0.7589}
top-12 {'llm_context_precision_with_reference': 0.5850, 'context_recall': 0.5773, 'nv_accuracy': 0.4437, 'faithfulness': 0.8273}


In [32]:
mmr_results_lst = []
for retriever, rag_chain in zip(mmr_rt_lst, mmr_rag_chains):
    eval_ds, retrieval_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    mmr_results_lst.append((result,retrieval_time))

Evaluating: 100%|██████████| 160/160 [08:26<00:00,  3.17s/it]


In [ ]:
for idx, (result, retrieval_time) in enumerate(mmr_results_lst):
    print(f'top-{k_values[idx]}', result)
    df = result.to_pandas()
    df["retrieval_time"] = retrieval_time
    df.to_csv(f"./evaluate_results/04_retrieval_test/mmr_retrieval/top_{k_values[idx]}.csv")

top-4 {'llm_context_precision_with_reference': 0.5910, 'context_recall': 0.4611, 'nv_accuracy': 0.4500, 'faithfulness': 0.7537}
top-6 {'llm_context_precision_with_reference': 0.5519, 'context_recall': 0.5513, 'nv_accuracy': 0.4688, 'faithfulness': 0.7825}
top-8 {'llm_context_precision_with_reference': 0.5597, 'context_recall': 0.5535, 'nv_accuracy': 0.4437, 'faithfulness': 0.7775}
top-10 {'llm_context_precision_with_reference': 0.5732, 'context_recall': 0.5868, 'nv_accuracy': 0.4688, 'faithfulness': 0.8368}
top-12 {'llm_context_precision_with_reference': 0.5361, 'context_recall': 0.6362, 'nv_accuracy': 0.4500, 'faithfulness': 0.7955}


Hybrid retriever

In [ ]:
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
 
# initialize BM25 and FAISS
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10
 
faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )


class HybridRetriever(BaseRetriever):
    
    def __init__(self, base_retriever, final_k=10):
        self.base_retriever = base_retriever
        self.final_k = final_k

    def invoke(self, query):
        docs = self.base_retriever.get_relevant_documents(query)

        # cut off top-10
        return docs[:self.final_k]


    def get_relevant_documents(self, query):
        return self.invoke(query)
 
# initialize Ensemble Retriever
hybrid_rt_lst = []
weights_lst = [[0.3,0.7],
               [0.4,0.6],
               [0.5,0.5],
               [0.6,0.4],
               [0.7,0.3],]

for weights in weights_lst:
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=weights
    )
    hybrid_retriever = HybridRetriever(ensemble_retriever, final_k=10)

    hybrid_rt_lst.append(hybrid_retriever)

In [ ]:
# save chains with Ensemble retriever
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

hybrid_rag_chains = []

for retriever in hybrid_rt_lst:
    context_retriever = RunnableLambda(lambda query, r=retriever: r.invoke(query))
    
    rag_chain = (
        {
          "context": context_retriever | RunnableLambda(format_docs), 
          "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    hybrid_rag_chains.append(rag_chain)

In [17]:
data_path = "evaluate_dataset/test_dataset.csv"
hybrid_results_lst = []

for retriever, rag_chain in zip(hybrid_rt_lst, hybrid_rag_chains):
    eval_ds, retrieval_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    hybrid_results_lst.append((result,retrieval_time))

/tmp/ipykernel_1003513/2139276848.py:22: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = self.base_retriever.get_relevant_documents(query)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Evaluating: 100%|██████████| 160/160 [15:21<00:00,  5.76s/it]


hybrid top-10

In [ ]:
for idx, (result, retrieval_time) in enumerate(hybrid_results_lst):
    print(f'bm25:{weights_lst[idx][0]}+mmr:{weights_lst[idx][1]}', result)
    df = result.to_pandas()
    df["retrieval_time"] = retrieval_time
    df.to_csv(f"./evaluate_results/04_retrieval_test/hybrid_retrieval/bm25_{weights_lst[idx][0]}_mmr:{weights_lst[idx][1]}.csv")

bm25:0.3+mmr:0.7 {'llm_context_precision_with_reference': 0.6276, 'context_recall': 0.6139, 'nv_accuracy': 0.4313, 'faithfulness': 0.8072}
bm25:0.4+mmr:0.6 {'llm_context_precision_with_reference': 0.6371, 'context_recall': 0.6262, 'nv_accuracy': 0.5125, 'faithfulness': 0.8249}
bm25:0.5+mmr:0.5 {'llm_context_precision_with_reference': 0.6144, 'context_recall': 0.6574, 'nv_accuracy': 0.4875, 'faithfulness': 0.8013}
bm25:0.6+mmr:0.4 {'llm_context_precision_with_reference': 0.5806, 'context_recall': 0.5829, 'nv_accuracy': 0.4375, 'faithfulness': 0.7955}
bm25:0.7+mmr:0.3 {'llm_context_precision_with_reference': 0.5883, 'context_recall': 0.5660, 'nv_accuracy': 0.4437, 'faithfulness': 0.8175}
